In [313]:
import pandas as pd
import json

In [314]:
df = pd.read_csv("../master_results.csv")
df.head(5)

,type,province,district,sub-district,unit_number,name,score
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ไทยทรัพย์ทวี,0
1,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อชาติไทย,6
2,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ใหม่,4
3,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,มิติใหม่,0
4,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมใจไทย,1


In [315]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22422 entries, 0 to 22421
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   type          22422 non-null  str  
 1   province      22422 non-null  str  
 2   district      22422 non-null  str  
 3   sub-district  22422 non-null  str  
 4   unit_number   22422 non-null  int64
 5   name          22422 non-null  str  
 6   score         22422 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1.2 MB


In [316]:
df.isnull().sum()

type            0
province        0
district        0
sub-district    0
unit_number     0
name            0
score           0
dtype: int64

In [317]:
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0)

In [318]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

In [319]:
d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

In [320]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)

14

In [321]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
sd2d['บ้านใหม่'] = 'วังเหนือ'

In [322]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [323]:
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    all_subdistrict.append(sub_dist)

In [324]:
df['province'] = 'ลำปาง'

In [325]:
df['province'].unique().tolist()

['ลำปาง']

In [326]:
wrong_districts = []
for wrong_district in df['district'].unique().tolist():
    if wrong_district not in all_distrcit and wrong_district != 'นอกเขต':
        wrong_districts.append(wrong_district)

wrong_districts

['แจ้งวัฒนะ',
 'แจ้งวิมล',
 'นางา',
 'อำเภองาว',
 'งาม',
 'องาว',
 'นางลิ้นจี่',
 'อำเภอวังเหนือ',
 'อำเภอเมืองปาน',
 'อำเภอเมืองลำปาง',
 'นางเงาน']

In [327]:
all_distrcit

['เมืองลำปาง', 'งาว', 'แจ้ห่ม', 'วังเหนือ', 'เมืองปาน', 'นอกเขต']

In [328]:
df[df['district'].isin(wrong_districts)]

,type,province,district,sub-district,unit_number,name,score
121,เขต,ลำปาง,แจ้งวัฒนะ,ชุดที่ 10,-1,นางสาววิภา วงศ์ประพันธ์,15.0
122,เขต,ลำปาง,แจ้งวัฒนะ,ชุดที่ 10,-1,นายภูมิศักดิ์ กุลจรุง,450.0
123,เขต,ลำปาง,แจ้งวัฒนะ,ชุดที่ 10,-1,นายพรเทพ พูลมาลา,23.0
124,เขต,ลำปาง,แจ้งวัฒนะ,ชุดที่ 10,-1,นายสมบูรณ์ อุปถากาล,14.0
125,เขต,ลำปาง,แจ้งวัฒนะ,ชุดที่ 10,-1,นายดำรงค์ เกิดมนูญ,40.0
...,...,...,...,...,...,...,...
22238,เขต,ลำปาง,นางเงาน,บ้านอ้อน,7,นายศรีพรหมอ หอมยก,-1.0
22239,เขต,ลำปาง,นางเงาน,บ้านอ้อน,7,นายสมบูรณ์ รูปสะอาด,-1.0
22240,เขต,ลำปาง,นางเงาน,บ้านอ้อน,7,นายดาชัย เอกปฏิพี,-1.0
22241,เขต,ลำปาง,นางเงาน,บ้านอ้อน,7,นายธนาธร โล่ห์สุนทร,-1.0


In [329]:
wrong2correct = dict()
scores = dict()
from rapidfuzz import process, utils
for text in wrong_districts:
    match = process.extractOne(text, all_distrcit, score_cutoff=50)
    
    if match:
        result, score, index = match
        wrong2correct[text] = result
        scores[text] = round(score,2)
        print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
    else:
        wrong2correct[text] = "can't find"
        scores[text] = 0
        print(f"OCR: '{text}' -> [Manual Review Needed]")

OCR: 'แจ้งวัฒนะ' -> Corrected: 'งาว' (Confidence: 60.00%)
OCR: 'แจ้งวิมล' -> Corrected: 'งาว' (Confidence: 60.00%)
OCR: 'นางา' -> Corrected: 'เมืองลำปาง' (Confidence: 60.00%)
OCR: 'อำเภองาว' -> Corrected: 'งาว' (Confidence: 90.00%)
OCR: 'งาม' -> Corrected: 'งาว' (Confidence: 66.67%)
OCR: 'องาว' -> Corrected: 'งาว' (Confidence: 85.71%)
OCR: 'นางลิ้นจี่' -> [Manual Review Needed]
OCR: 'อำเภอวังเหนือ' -> Corrected: 'วังเหนือ' (Confidence: 90.00%)
OCR: 'อำเภอเมืองปาน' -> Corrected: 'เมืองปาน' (Confidence: 90.00%)
OCR: 'อำเภอเมืองลำปาง' -> Corrected: 'เมืองลำปาง' (Confidence: 90.00%)
OCR: 'นางเงาน' -> Corrected: 'งาว' (Confidence: 60.00%)


In [330]:
df[df['district'] == wrong_districts[0]]['district'] = 'นอกเขต'

C:\Users\USER\AppData\Local\Temp\ipykernel_23124\3901947573.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df['district'] == wrong_districts[0]]['district'] = 'นอกเขต'


In [331]:
df.loc[df['district'] == wrong_districts[1],'district'] = 'นอกเขต'
df[df['sub-district'].str.contains("ชุดที่")]['sub-district'].unique().tolist()

['ชุดที่ 1',
 'ชุดที่ 10',
 'ชุดที่ 11',
 'ชุดที่ 12',
 'ชุดที่ 13',
 'ชุดที่ 2',
 'ชุดที่ 3',
 'ชุดที่ 4',
 'ชุดที่ 5',
 'ชุดที่ 6',
 'ชุดที่ 7',
 'ชุดที่ 8',
 'ชุดที่ 9']

In [332]:
df_test = df.copy()
df_test['district'] = df_test.apply(lambda row: sd2d.get(row['sub-district'], row['district']), axis=1)
wrong_districts = [dist for dist in df_test['district'].unique().tolist() if dist not in all_distrcit + ['นอกเขต']]

In [333]:
df_test['district'].unique().tolist()

['นอกเขต',
 'งาว',
 'อำเภองาว',
 'วังเหนือ',
 'เมืองปาน',
 'อำเภอเมืองปาน',
 'เมืองลำปาง',
 'แจ้ห่ม']

In [334]:
from rapidfuzz import process
def automate_edit(feature, check, wrongs, df):
    wrong2correct = dict()
    scores = dict()
    for text in wrongs:
        match = process.extractOne(text, check, score_cutoff=50)

        if match:
            result, score, index = match
            wrong2correct[text] = result
            scores[text] = round(score,2)
            if scores[text] >= 0.8:
                df.loc[df_test[feature] == text, feature] = result
            print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
        else:
            wrong2correct[text] = "can't find"
            scores[text] = 0
            print(f"OCR: '{text}' -> [Manual Review Needed]")
    return wrong2correct, scores

In [335]:
wrong2correct, scores = automate_edit('district', all_distrcit, wrong_districts, df_test)

OCR: 'อำเภองาว' -> Corrected: 'งาว' (Confidence: 90.00%)
OCR: 'อำเภอเมืองปาน' -> Corrected: 'เมืองปาน' (Confidence: 90.00%)


In [336]:
df_test['district'].unique().tolist()

['นอกเขต', 'งาว', 'วังเหนือ', 'เมืองปาน', 'เมืองลำปาง', 'แจ้ห่ม']

In [337]:
df_test[df_test['sub-district'] == 'บ้านใหม่']

,type,province,district,sub-district,unit_number,name,score
6525,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,นางสาววิชุดา ว่องวัฒนวิโรจน์,3.0
6526,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,นางสาวสุภิภา กุศลจูง,134.0
6527,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,นายศรีพรหมอ หอมยก,2.0
6528,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,นายสมบูรณ์ รูปสะอาด,1.0
6529,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,นายดาซัย เอกปฐพี,50.0
...,...,...,...,...,...,...,...
7852,บช,ลำปาง,วังเหนือ,บ้านใหม่,4,ไทยพิทักษ์ธรรม,1.0
7853,บช,ลำปาง,วังเหนือ,บ้านใหม่,4,ความหวังใหม่,0.0
7854,บช,ลำปาง,วังเหนือ,บ้านใหม่,4,ไทยรวมไทย,0.0
7855,บช,ลำปาง,วังเหนือ,บ้านใหม่,4,เพื่อบ้านเมือง,1.0


In [338]:
df = df_test

In [339]:
df_test = df.copy()

In [340]:
wrong_subdistricts = [sd for sd in df_test['sub-district'].unique().tolist() if sd not in all_subdistrict]
wrong_subdistricts

['บ้านหวอด',
 'เจ้ซ้อน',
 'เจริญยุทธ',
 'เจ้าซ้อน',
 'แจ้งขึ้น',
 'เจ็ดสิบ',
 'ทุ่งกว้าง']

In [341]:
wrong2correct, scores = automate_edit('sub-district', all_subdistrict, wrong_subdistricts, df_test)

OCR: 'บ้านหวอด' -> Corrected: 'บ้านหวด' (Confidence: 93.33%)
OCR: 'เจ้ซ้อน' -> Corrected: 'แจ้ซ้อน' (Confidence: 85.71%)
OCR: 'เจริญยุทธ' -> [Manual Review Needed]
OCR: 'เจ้าซ้อน' -> Corrected: 'แจ้ซ้อน' (Confidence: 80.00%)
OCR: 'แจ้งขึ้น' -> Corrected: 'แจ้ซ้อน' (Confidence: 66.67%)
OCR: 'เจ็ดสิบ' -> [Manual Review Needed]
OCR: 'ทุ่งกว้าง' -> Corrected: 'ทุ่งกว๋าว' (Confidence: 77.78%)


In [342]:
wrong_subdistricts = [sd for sd in df_test['sub-district'].unique().tolist() if sd not in all_subdistrict]

Manual check above sub-district

In [343]:
df_test.loc[df_test['sub-district'] == wrong_subdistricts[0], 'sub-district'] = 'แจ้ซ้อน'

In [344]:
df_test.loc[df_test['sub-district'] == wrong_subdistricts[1], 'sub-district'] = 'แจ้ซ้อน'

In [345]:
wrong_subdistricts = [sd for sd in df_test['sub-district'].unique().tolist() if sd not in all_subdistrict]
wrong_subdistricts

[]

In [346]:
summary = pd.read_csv("../master_summary.csv", index_col=0)
# valid_cards = summary.groupby(['district','sub-district','unit_number'])
# valid_cards
summary

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,663,8,29,-1
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,605,32,63,-1
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,668,8,24,-1
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,616,28,56,-1
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,664,6,30,-1
...,...,...,...,...,...,...,...,...,...,...
705,เขต,ลำปาง,แจ้ห่ม,วิเชตนคร,11,320,209,17,24,70
706,เขต,ลำปาง,แจ้ห่ม,บ้านสา,8,320,215,11,9,85
707,เขต,ลำปาง,งาว,หลวงเหนือ,3,560,325,13,19,203
708,เขต,ลำปาง,นอกเขต,ชุดที่ 5,-1,700,620,26,54,-1


In [352]:
s = 0
for dist in d.keys():
    for sd in d[dist].keys():
        for i in range(1,d[dist][sd]+1):
            if dist == 'นอกเขต':
                e = len(df_test[(df_test['district'] == dist) & (df_test['sub-district'] == f'ชุดที่ {i}') & (df_test['unit_number'] == -1)])
                if e != 64:
                    print(f"อำเภอ,เขต,หน่วย: {dist},{sd},{i} ไม่ครบ 64 เพราะมีแค่: {e}")
                    s+=(64-e)
            else:
                e = len(df_test[(df_test['district'] == dist) & (df_test['sub-district'] == sd) & (df_test['unit_number'] == i)])
                if e != 64:
                    print(f"อำเภอ,เขต,หน่วย: {dist},{sd},{i} ไม่ครบ 64 เพราะมีแค่: {e}")
                    s+=(64-e)

อำเภอ,เขต,หน่วย: งาว,บ้านโป่ง,10 ไม่ครบ 64 เพราะมีแค่: 30
อำเภอ,เขต,หน่วย: งาว,บ้านโป่ง,11 ไม่ครบ 64 เพราะมีแค่: 17
อำเภอ,เขต,หน่วย: งาว,บ้านหวด,5 ไม่ครบ 64 เพราะมีแค่: 40
อำเภอ,เขต,หน่วย: งาว,บ้านหวด,6 ไม่ครบ 64 เพราะมีแค่: 17
อำเภอ,เขต,หน่วย: แจ้ห่ม,บ้านสา,6 ไม่ครบ 64 เพราะมีแค่: 71
อำเภอ,เขต,หน่วย: แจ้ห่ม,บ้านสา,8 ไม่ครบ 64 เพราะมีแค่: 57
อำเภอ,เขต,หน่วย: แจ้ห่ม,เมืองมาย,2 ไม่ครบ 64 เพราะมีแค่: 54
อำเภอ,เขต,หน่วย: แจ้ห่ม,เมืองมาย,4 ไม่ครบ 64 เพราะมีแค่: 15
อำเภอ,เขต,หน่วย: แจ้ห่ม,วิเชตนคร,8 ไม่ครบ 64 เพราะมีแค่: 30
อำเภอ,เขต,หน่วย: วังเหนือ,ทุ่งฮั้ว,6 ไม่ครบ 64 เพราะมีแค่: 63
อำเภอ,เขต,หน่วย: วังเหนือ,ทุ่งฮั้ว,10 ไม่ครบ 64 เพราะมีแค่: 40
อำเภอ,เขต,หน่วย: วังเหนือ,ทุ่งฮั้ว,11 ไม่ครบ 64 เพราะมีแค่: 54
อำเภอ,เขต,หน่วย: เมืองปาน,เมืองปาน,2 ไม่ครบ 64 เพราะมีแค่: 52


In [353]:
print(s)

292


Mannual Check and Update

In [354]:
list_check  =[]
for type in ['เขต','บช']:
    for dist in d.keys():
        for sd in d[dist].keys():
            for i in range(1,d[dist][sd]+1):
                if dist == 'นอกเขต':
                    e = summary[(summary['district'] == dist) & (summary['sub-district'] == f'ชุดที่ {i}') & (summary['unit_number'] == -1) & (summary['type'] == type)]['valid_ballots'].values
                    t = df_test[(df_test['district'] == dist) & (df_test['sub-district'] == f'ชุดที่ {i}') & (df_test['unit_number'] == -1) & (df_test['type'] == type)]['score'].sum()
                    if e != t:
                        text = f"อำเภอ,เขต,หน่วย: {dist},{sd},{i} จำนวนบัตร validไม่ตรงกับ {e} เพราะมีแค่ {t}"
                        print(f"type อำเภอ,เขต,หน่วย:{type}, {dist},{sd},{i} จำนวนบัตร validไม่ตรงกับ {e} เพราะมีแค่ {t}")
                        list_check.append(text)
                else:
                    e = summary[(summary['district'] == dist) & (summary['sub-district'] == sd) & (summary['unit_number'] == i) & (summary['type'] == type)]['valid_ballots'].values
                    t = df_test[(df_test['district'] == dist) & (df_test['sub-district'] == sd) & (df_test['unit_number'] == i) & (df_test['type'] == type)]['score'].sum()
                    if e != t:
                        text = f"อำเภอ,เขต,หน่วย: {dist},{sd},{i} จำนวนบัตร validไม่ตรงกับ {e} เพราะมีแค่ {t}"
                        print(f"type อำเภอ,เขต,หน่วย:{type}, {dist},{sd},{i} จำนวนบัตร validไม่ตรงกับ {e} เพราะมีแค่ {t}")
                        list_check.append(text)

type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านแลง,3 จำนวนบัตร validไม่ตรงกับ [335] เพราะมีแค่ 380.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านแลง,4 จำนวนบัตร validไม่ตรงกับ [317] เพราะมีแค่ 434.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านแลง,7 จำนวนบัตร validไม่ตรงกับ [126] เพราะมีแค่ 70.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านแลง,8 จำนวนบัตร validไม่ตรงกับ [334] เพราะมีแค่ 364.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านแลง,11 จำนวนบัตร validไม่ตรงกับ [406] เพราะมีแค่ 384.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านเสด็จ,2 จำนวนบัตร validไม่ตรงกับ [219] เพราะมีแค่ 221.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านเสด็จ,7 จำนวนบัตร validไม่ตรงกับ [306] เพราะมีแค่ 366.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านเสด็จ,8 จำนวนบัตร validไม่ตรงกับ [-1] เพราะมีแค่ 302.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านเสด็จ,9 จำนวนบัตร validไม่ตรงกับ [302] เพราะมีแค่ 180.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้านเสด็จ,10 จำนวนบัตร validไม่ตรงกับ [180] เพราะมีแค่ 381.0
type อำเภอ,เขต,หน่วย:เขต, เมืองลำปาง,บ้า

In [355]:
len(list_check)

442